# STAGE 2 — register-transfer BanglaT5

**Task:** external Bengali draft → competition-register doctor answer.
`seed 11 · lr 1e-3 · warmup 200 · 2 epochs · eff-batch 64 · eval every 250 steps`

## Why this exists

The competition `id` is a row index into ChatDoctor / HealthCareMagic-100k
([github.com/Kent0n-Li/ChatDoctor](https://github.com/Kent0n-Li/ChatDoctor)). Our Bengali
translation of that corpus is keyed on the same index, so **every competition row has an
independent Bengali translation of the same doctor answer** — 1,000/1,000 test rows covered.

Predicting that translation verbatim scores far above anything we have trained:

| Prediction source | Token F1 | ROUGE-L | pred LB |
|---|---|---|---|
| Best fine-tune (arm C) | 0.2576 | 0.1776 | 0.5800 |
| Constant string *(public #1)* | 0.2669 | 0.1564 | 0.57849 |
| **External draft, verbatim** | **0.5900** | **0.5398** | **0.7522** |
| **Draft + brand/greeting regex** | **0.5984** | **0.5482** | **0.7564** |

**But a lookup is not model output** (Rules §8) and cannot satisfy Phase 2 reproducibility — the
same trap the constant-string probe is in. So this notebook trains a model to do the part that is
actually left: convert the external translator's register into the competition translator's.

**The gap is large, systematic, and exactly what the metric rewards** (dev, n=5,000):

| | external draft | competition references |
|---|---|---|
| opens with `হেলো` | **0.06%** | 76.62% |
| contains `নাসেনিয়া` | **0.00%** | 49.98% |
| mean tokens | 101.0 | 100.2 |

Three regexes recover +0.0084 Token F1. A model trained on **101,737 aligned pairs** should recover
much more — and its output is genuinely generated, reproducible, and Phase-2-legal.

**Target to beat: Token F1 0.5984 / ROUGE-L 0.5482.** Below that, the model is worse than a regex
and we ship the regex logic inside a model instead.

## Disclosure (Rules §2.6.a — mandatory in the Phase 2 write-up)
Source **https://github.com/Kent0n-Li/ChatDoctor** — public repo, datasets on open Google Drive
links, no registration, no cost. Licence: code Apache-2.0; datasets *"for academic research only;
any commercial use and clinical use is prohibited"* — compatible with this Community/Kudos-only
competition, whose own data is CC BY-NC 4.0. The Bengali translation is our own derived artifact.

⚠️ **Accelerator = GPU T4**, **Internet = On**, ~6 h.

In [ ]:
# ══ 1 — hardware gate ═══════════════════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]} — Kaggle's PyTorch has no sm_60 kernels. Use GPU T4."
print("✅ hardware OK")

In [ ]:
# ══ 2 — pinned libs (trap #00: Kaggle ships transformers 5.0.0, which breaks T5) ══
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers", transformers.__version__, "| normalizer OK:", normalize("হেলো,  নাসেনিয়া ডকে"))

In [ ]:
# ══ 3 — locate code, competition data, external translation ═════════════════
import glob, os, shutil, sys
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/05_build_transfer.py", recursive=True)
assert hits, ("stale or missing code dataset — need tashintahir/nascenia-code with "
              "05_build_transfer.py. Add Input -> Datasets -> newest version.")
CODE = os.path.dirname(hits[0])

raw = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert raw, "Attach Add Input -> Datasets -> tashintahir/nascenia-data"
RAW = os.path.dirname(raw[0])

hcm = glob.glob("/kaggle/input/**/hcm_bn.csv", recursive=True)
assert hcm, "Attach Add Input -> Datasets -> tashintahir/nascenia-hcm-bn"
HCM = hcm[0]

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")

# Kaggle DataLoader-worker deadlock after eval/checkpointing; patch the writable copy only.
from pathlib import Path
tp = Path("/kaggle/working/code/02_train_t5.py"); s = tp.read_text(encoding="utf-8")
if "dataloader_num_workers=2," in s:
    s = s.replace("dataloader_num_workers=2,", "dataloader_num_workers=0,", 1)
s = s.replace("logging_steps=100,", "logging_steps=25,", 1)
tp.write_text(s, encoding="utf-8")
sys.path.insert(0, "/kaggle/working/code")
print("CODE:", CODE, "\nRAW :", RAW, "\nHCM :", HCM)

In [ ]:
# ══ 4 — frozen dev split (seed 42 governs the SPLIT — never change it) ══════
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

In [ ]:
# ══ 5 — build the register-transfer dataset ═════════════════════════════════
# input = external Bengali draft, target = competition-register answer.
# Asserts 100% dev/test coverage: a row with no draft cannot be predicted by this model
# at all, so a coverage hole would be a silent scoring hole.
!cd /kaggle/working/code && python 05_build_transfer.py --proc /kaggle/working/processed --hcm "{HCM}" --out /kaggle/working/xfer

In [ ]:
# ══ 6 — sanity: what the model will actually see ════════════════════════════
import pandas as pd
tr = pd.read_parquet("/kaggle/working/xfer/train.parquet")
print(f"train {len(tr)} rows\n")
print("INPUT  (external draft):\n ", tr['input'].iloc[0][:300], "\n")
print("TARGET (competition register):\n ", tr['output'].iloc[0][:300], "\n")
for name, col in (("input ", tr['input']), ("target", tr['output'])):
    print(f"{name}: opens হেলো {col.str.startswith('হেলো').mean()*100:5.2f}%  "
          f"has নাসেনিয়া {col.str.contains('নাসেনিয়া').mean()*100:5.2f}%  "
          f"tokens {col.str.split().str.len().mean():.1f}")
print("\n^ the gap between these two rows is what the model has to learn")

In [ ]:
# ══ 7 — SMOKE TEST (~3 min) — validates the pipeline before 6 GPU-hours ═════
import subprocess, shlex, os
RUN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
           "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1"}
cmd = ("python 02_train_t5.py --data-dir /kaggle/working/xfer --out-dir /kaggle/working/runs "
       "--smoke --max-train 1000 --epochs 1 --batch-size 8 --grad-accum 2 "
       "--eval-subset 100 --eval-steps 20 --eval-batch-size 8 --gen-num-beams 2 "
       "--no-bertscore-eval --run-name smoke")
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code", env=RUN_ENV)
assert r.returncode == 0, "SMOKE FAILED — do not start the full run"
print("\n✅ smoke passed")

In [ ]:
# ══ 8 — FULL RUN ════════════════════════════════════════════════════════════
# lr 1e-3 is the sweep winner (arm C). eval every 250 steps, NOT 500: arm C peaked at
# step 2000 and then DECLINED while loss kept falling, so the best checkpoint is not the
# last one and a coarse eval grid can miss the peak entirely.
import subprocess, shlex, os, time
RUN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
           "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1"}

SEED = 11
cmd = (f"python 02_train_t5.py "
       f"--data-dir /kaggle/working/xfer --out-dir /kaggle/working/runs "
       f"--model csebuetnlp/banglat5 --seed {SEED} "
       f"--epochs 2 --lr 1e-3 --warmup 200 --optim adafactor "
       f"--batch-size 8 --grad-accum 8 --eval-batch-size 8 "
       f"--max-source-len 384 --max-target-len 256 "
       f"--eval-subset 300 --eval-steps 250 "
       f"--gen-num-beams 4 --gen-min-new-tokens 80 "
       f"--precision auto --no-group-by-length --run-name banglat5_xfer_seed{SEED}")
print(cmd + "\n" + "=" * 70, flush=True)
t0 = time.time()
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code", env=RUN_ENV)
print(f"\nexit {r.returncode} after {(time.time()-t0)/60:.1f} min")
assert r.returncode == 0, "training failed — see traceback above"

In [ ]:
# ══ 9 — run record ══════════════════════════════════════════════════════════
import json, glob
for f in sorted(glob.glob("/kaggle/working/runs/*/run.json")):
    if "/smoke/" in f: continue
    print(f"\n=== {f} ===")
    print(json.dumps(json.load(open(f)), indent=2, ensure_ascii=False))

---
## How to judge this run

**Bar: Token F1 0.5984 / ROUGE-L 0.5482** — what the draft plus three regexes already achieves.

| Result | Meaning |
|---|---|
| **> 0.60 Token F1** | The model is adding real register conversion. Ship it — genuine model output, Phase-2-legal, above every alternative we have. |
| **≈ 0.598** | The model learned to copy the draft and little else. Still legitimate model output, but no gain over the regex — prefer the simpler pipeline. |
| **< 0.55** | The model is *destroying* information present in the draft. Investigate before spending another run. |

Watch `eval_pred_tokens` (~100 expected) and check whether outputs pick up `হেলো` / `নাসেনিয়া`.

**Then:** save `best/` as a Kaggle Dataset, record in `LOCAL_EXPERIMENTS.md`, and build the
submission notebook with the same known-good-number assert pattern as the arm notebooks.